In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from frag_gnn.runner import load_config, init_dataset, init_dataloader
from frag_gnn.pl_model import FragGNNPL
from frag_gnn.dataset import SpecMolFragDataset
from frag_gnn.utils.misc_utils import scatter_reduce, scatter_argmax, scatter_argtopk, EPS, booltype, scatter_masked_softmax, scatter_logsumexp, safelog
from frag_gnn.utils.script_utils import init_wandb_ckpt, run_inference, select_model_vals, sparse_kl, log_mean, log_std
from frag_gnn.loss import sparse_conditional_entropy_fn

import numpy as np
import torch as th
import pandas as pd
import argparse	
import wandb
import os
from pprint import pprint

In [3]:
baseline_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_baseline_test_False/vals.pkl"
baseline_vals = th.load(baseline_vals_fp)

In [4]:
baseline_vals["entropyv3_scaffold_d3_baseline_ens_s00"].keys()

dict_keys(['unique_id', 'smiles', 'mean_loss', 'total_loss', 'total_weight', 'spec_per_group', 'spec_per_mol', 'group_per_mol', 'true_mzs', 'true_logprobs', 'true_batch_idxs', 'true_prec_mzs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'pred_formula_logprobs', 'pred_formula_formula_idxs', 'pred_formula_batch_idxs', 'pred_node_logprobs', 'pred_node_node_idxs', 'pred_node_batch_idxs', 'pred_node_formula_logprobs', 'pred_formula_node_logprobs', 'pred_joint_logprobs', 'pred_joint_node_idxs', 'pred_joint_formula_idxs', 'pred_joint_batch_idxs', 'pred_joint_h_counts', 'pred_joint_h_idxs', 'pred_null_formula_logprob', 'pred_oos_logprobs', 'pred_h_counts', 'pred_h_batch_idxs', 'pred_h_logprobs', 'pred_nb_node_logprobs', 'pred_nb_node_formula_logprobs', 'pred_nb_formula_node_logprobs', 'pred_nb_node_node_logprobs', 'pred_nb_node_node_idxs', 'pred_nb_node_batch_idxs', 'pred_nb_joint_logprobs', 'pred_nb_joint_node_idxs', 'pred_nb_joint_formula_idxs', 'pred_nb_joint_batch_idxs', 'pred_nb_node_

In [5]:
low_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_posfn_test_False/vals.pkl"
low_vals = th.load(low_vals_fp)

In [6]:
low_vals["entropyv3_scaffold_d3_posfn_ens_s00"].keys()

dict_keys(['unique_id', 'smiles', 'mean_loss', 'total_loss', 'total_weight', 'spec_per_group', 'spec_per_mol', 'group_per_mol', 'true_mzs', 'true_logprobs', 'true_batch_idxs', 'true_prec_mzs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'pred_formula_logprobs', 'pred_formula_formula_idxs', 'pred_formula_batch_idxs', 'pred_node_logprobs', 'pred_node_node_idxs', 'pred_node_batch_idxs', 'pred_node_formula_logprobs', 'pred_formula_node_logprobs', 'pred_joint_logprobs', 'pred_joint_node_idxs', 'pred_joint_formula_idxs', 'pred_joint_batch_idxs', 'pred_joint_h_counts', 'pred_joint_h_idxs', 'pred_null_formula_logprob', 'pred_oos_logprobs', 'pred_h_counts', 'pred_h_batch_idxs', 'pred_h_logprobs', 'pred_nb_node_logprobs', 'pred_nb_node_formula_logprobs', 'pred_nb_formula_node_logprobs', 'pred_nb_node_node_logprobs', 'pred_nb_node_node_idxs', 'pred_nb_node_batch_idxs', 'pred_nb_joint_logprobs', 'pred_nb_joint_node_idxs', 'pred_nb_joint_formula_idxs', 'pred_nb_joint_batch_idxs', 'pred_nb_node_

In [7]:
high_vals_fp = "../inference_outputs/entropyv3/scaffold_d3_negfn_test_False/vals.pkl"
high_vals = th.load(high_vals_fp)

In [8]:
def ensemble_vals(vals, nb=True):

    select_keys = [
        "pred_formula_logprobs",
        "pred_formula_batch_idxs",
        "pred_formula_formula_idxs",
    ]
    if nb:
        select_keys += [
            "pred_nb_joint_logprobs",
            "pred_nb_formula_node_logprobs",
            "pred_nb_joint_formula_idxs",
            "pred_nb_joint_batch_idxs",
            "pred_nb_joint_node_idxs"
        ]
    else:
        select_keys += [
            "pred_joint_logprobs",
            "pred_formula_node_logprobs",
            "pred_joint_formula_idxs",
            "pred_joint_batch_idxs",
            "pred_joint_node_idxs",
    ]

    select_d = select_model_vals(vals,select_keys,stack_dim=0)
    # formula stuff
    ens_formula_logprobs = log_mean(select_d["pred_formula_logprobs"],dim=0)
    ens_formula_batch_idxs = select_d["pred_formula_batch_idxs"][0]
    ens_formula_formula_idxs = select_d["pred_formula_formula_idxs"][0]
    
    ens_d = {
        "ens_formula_logprobs": ens_formula_logprobs,
        "ens_formula_batch_idxs": ens_formula_batch_idxs,
        "ens_formula_formula_idxs": ens_formula_formula_idxs,
    }
    
    if nb:
        
        # non-iso joint stuff
        ens_nb_joint_logprobs = log_mean(select_d["pred_nb_joint_logprobs"],dim=0)
        ens_nb_joint_formula_idxs = select_d["pred_nb_joint_formula_idxs"][0]
        ens_nb_joint_batch_idxs = select_d["pred_nb_joint_batch_idxs"][0]
        ens_nb_joint_node_idxs = select_d["pred_nb_joint_node_idxs"][0]
        ens_nb_formula_node_logprobs = ens_nb_joint_logprobs - ens_formula_logprobs[ens_nb_joint_formula_idxs]
        ens_nb_formula_node_logprobs = th.clamp(ens_nb_formula_node_logprobs, max=0.)
        ens_nb_formula_node_lse = scatter_logsumexp(ens_nb_formula_node_logprobs, ens_nb_joint_formula_idxs, dim_size=ens_formula_formula_idxs.shape[0])
    
        ens_d.update({
            "ens_nb_joint_logprobs": ens_nb_joint_logprobs,
            "ens_nb_joint_formula_idxs": ens_nb_joint_formula_idxs,
            "ens_nb_joint_batch_idxs": ens_nb_joint_batch_idxs,
            "ens_nb_joint_node_idxs": ens_nb_joint_node_idxs,
            "ens_nb_formula_node_logprobs": ens_nb_formula_node_logprobs,
            "ens_nb_formula_node_lse": ens_nb_formula_node_lse
        })

    else:

        # iso joint stuff
        ens_joint_logprobs = log_mean(select_d["pred_joint_logprobs"],dim=0)
        ens_joint_formula_idxs = select_d["pred_joint_formula_idxs"][0]
        ens_joint_batch_idxs = select_d["pred_joint_batch_idxs"][0]
        ens_joint_node_idxs = select_d["pred_joint_node_idxs"][0]
        ens_formula_node_logprobs = ens_joint_logprobs - ens_formula_logprobs[ens_joint_formula_idxs]
        ens_formula_node_logprobs = th.clamp(ens_formula_node_logprobs, max=0.)
        ens_formula_node_lse = scatter_logsumexp(ens_formula_node_logprobs, ens_joint_formula_idxs, dim_size=ens_formula_formula_idxs.shape[0])
    
        ens_d.update({
            "ens_joint_logprobs": ens_joint_logprobs,
            "ens_joint_formula_idxs": ens_joint_formula_idxs,
            "ens_joint_batch_idxs": ens_joint_batch_idxs,
            "ens_joint_node_idxs": ens_joint_node_idxs,
            "ens_formula_node_logprobs": ens_formula_node_logprobs,
            "ens_formula_node_lse": ens_formula_node_lse,
        })
    
    return ens_d

In [9]:
baseline_nb_ens_vals = ensemble_vals(baseline_vals, nb=True)


In [10]:
low_nb_ens_vals = ensemble_vals(low_vals, nb=True)

In [11]:
high_nb_ens_vals = ensemble_vals(high_vals, nb=True)

In [12]:
def get_topk_formula_node_idxs(ens_vals, k, nb=True, filter_thresh=0.05):

    ens_formula_logprobs = ens_vals["ens_formula_logprobs"]
    ens_formula_mask = ens_formula_logprobs > np.log(filter_thresh)

    if nb:
        prefix = "nb_"
    else:
        prefix = ""

    ens_counts = scatter_reduce(
        src=th.ones_like(ens_vals[f"ens_{prefix}formula_node_logprobs"], dtype=th.long),
        index=ens_vals[f"ens_{prefix}joint_formula_idxs"],
        reduce="sum",
        dim_size=th.max(ens_vals[f"ens_{prefix}joint_formula_idxs"])+1
    )

    ens_nb_formula_node_argtopk = scatter_argtopk(
        ens_vals[f"ens_{prefix}formula_node_logprobs"],
        ens_vals[f"ens_{prefix}joint_formula_idxs"],
        ens_vals[f"ens_{prefix}joint_node_idxs"],
        k,
        dim_size=ens_formula_mask.shape[0],
        return_max=False
    )

    return ens_formula_mask, ens_counts, ens_nb_formula_node_argtopk

    

In [13]:
k = 1

In [14]:
baseline_nb_mask, baseline_nb_counts, baseline_nb_argtopk = get_topk_formula_node_idxs(baseline_nb_ens_vals, k, nb=True)
print((baseline_nb_counts[baseline_nb_mask] < k).float().mean())
print((baseline_nb_argtopk[baseline_nb_mask] == -1).any(dim=1).float().mean())

tensor(0.)
tensor(0.)


In [15]:
low_nb_mask, low_nb_counts, low_nb_argtopk = get_topk_formula_node_idxs(low_nb_ens_vals, k, nb=True)
print((low_nb_counts[low_nb_mask] < k).float().mean())
print((low_nb_argtopk[low_nb_mask] == -1).any(dim=1).float().mean())

tensor(0.)
tensor(0.)


In [16]:
high_nb_mask, high_nb_counts, high_nb_argtopk = get_topk_formula_node_idxs(high_nb_ens_vals, k, nb=True)
print((high_nb_counts[high_nb_mask] < k).float().mean())
print((high_nb_argtopk[high_nb_mask] == -1).any(dim=1).float().mean())

tensor(0.)
tensor(0.)


In [17]:
def compare_all(masks, argtopks):

    num_ens = len(masks)
    for i in range(num_ens):
        for j in range(i):
            mask_i = masks[i]
            mask_j = masks[j]
            argtopk_i = argtopks[i]
            argtopk_j = argtopks[j]
            mask_ij = mask_i & mask_j
            equal_ij = th.all(argtopk_i[mask_ij] == argtopk_j[mask_ij], dim=1)
            print(i, j, equal_ij.float().mean())

In [18]:
compare_all(
    [baseline_nb_mask, low_nb_mask, high_nb_mask],
    [baseline_nb_argtopk, low_nb_argtopk, high_nb_argtopk]
)

1 0 tensor(0.9897)
2 0 tensor(0.9867)
2 1 tensor(0.9840)


In [19]:
# baseline_mask, baseline_counts, baseline_argtopk = get_topk_formula_node_idxs(baseline_nb_ens_vals, k, nb=False)
# print((baseline_counts[baseline_mask] < k).float().mean())
# print((baseline_argtopk[baseline_mask] == -1).any(dim=1).float().mean())